## Imports

In [ ]:
import polars as pl
from pathlib import Path

# Data Path

In [ ]:
data_folder = Path("../data")

## Load Dataset

In [ ]:
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview dataset
sales_df.head()

## Total Revenue and Transactions by City

In [ ]:
city_performance = sales_df.group_by(["city"]).agg (
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue"),
    pl.col("price").mean().round().alias("average_order_value")
).sort("total_revenue", descending=True)

city_performance

## Festival Sales by City

In [ ]:
festival_city = sales_df.group_by(["festival", "city"]).agg (
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue")
).sort(["festival", "total_revenue"], descending=[False, True])

festival_city.head(10)

# City Revenue Share

In [ ]:
city_performance = city_performance.with_columns(
   (
     pl.col("total_revenue")/pl.col("total_revenue").sum() * 100 
   ).round(1).alias("revenue_share_pct")
)

city_performance.sort("revenue_share_pct",descending = True)

In [ ]:
festival_city = sales_df.group_by(["festival", "city"]).agg(
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue")
).sort(["festival", "total_revenue"], descending=[False, True])

festival_city.head(10)

# Save Outputs

In [ ]:
city_performance.write_parquet(data_folder / "city_performance.parquet")
festival_city.write_parquet(data_folder / "festival_city.parquet")